# # Day 20 — EMBER 2018 Dataset EDA

In [3]:
# %%
# --- Cell 1: imports ---
import os
import ember              # the elastic/ember library, gives us the vectorization + readers
import numpy as np
import pandas as pd

DATA_DIR = r"C:\My_Files\Work\Projects\Malware_Detection\strazh\data\ember2018"   # wherever you extracted the tar.bz2 to

In [4]:
# %%
# --- Cell 2: build the vectorized feature matrices (one-time, cached to disk) ---
# This reads the raw JSONL records EMBER ships and flattens each one into
# a fixed-length float32 vector, saving X_train.dat / y_train.dat / X_test.dat / y_test.dat
# Skips work automatically if the .dat files already exist.
ember.create_vectorized_features(DATA_DIR)

# Also build a lightweight metadata table (sha256, label, appeared date, subtype)
# separate from the big numeric matrix — cheaper to explore label balance with this.
ember.create_metadata(DATA_DIR)

Vectorizing training set


100%|██████████| 800000/800000 [10:23:21<00:00, 21.39it/s]    


Vectorizing test set


100%|██████████| 200000/200000 [01:14<00:00, 2697.16it/s]


,sha256,appeared,label,avclass,subset
0,0abb4fda7d5b13801d63bee53e5e256be43e141faa077a...,2006-12,0,,train
1,c9cafff8a596ba8a80bafb4ba8ae6f2ef3329d95b85f15...,2007-01,0,,train
2,eac8ddb4970f8af985742973d6f0e06902d42a3684d791...,2007-02,0,,train
3,7f513818bcc276c531af2e641c597744da807e21cc1160...,2007-02,0,,train
4,ca65e1c387a4cc9e7d8a8ce12bf1bcf9f534c9032b9d95...,2007-02,0,,train
...,...,...,...,...,...
999995,e033bc4967ce64bbb5cafdb234372099395185a6e0280c...,2018-12,1,zbot,test
999996,c7d16736fd905f5fbe4530670b1fe787eb12ee86536380...,2018-12,1,flystudio,test
999997,0020077cb673729209d88b603bddf56b925b18e682892a...,2018-12,0,,test
999998,1b7e7c8febabf70d1c17fe3c7abf80f33003581c380f28...,2018-12,0,,test


In [5]:
# %%
# --- Cell 3: load everything back in ---
X_train, y_train, X_test, y_test = ember.read_vectorized_features(DATA_DIR)
metadata_df = ember.read_metadata(DATA_DIR)

print("X_train shape:", X_train.shape)   # (rows, 2381 features)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)


X_train shape: (800000, 2381)
y_train shape: (800000,)
X_test shape: (200000, 2381)


In [6]:
# %%
# --- Cell 4: label distribution ---
# EMBER encodes: 1 = malicious, 0 = benign, -1 = unlabeled
train_labels = pd.Series(y_train).value_counts().sort_index()
test_labels = pd.Series(y_test).value_counts().sort_index()

print("Train label counts:\n", train_labels)
print("\nTest label counts:\n", test_labels)

# what fraction of TRAIN rows are actually usable (labeled) for supervised training?
labeled_mask = y_train != -1
print(f"\nLabeled train rows: {labeled_mask.sum()} / {len(y_train)} "
      f"({labeled_mask.mean():.1%})")

# class balance among only the labeled rows — this is what actually matters for training
labeled_balance = pd.Series(y_train[labeled_mask]).value_counts(normalize=True)
print("\nClass balance (labeled only):\n", labeled_balance)

Train label counts:
 -1.0    200000
 0.0    300000
 1.0    300000
Name: count, dtype: int64

Test label counts:
 0.0    100000
1.0    100000
Name: count, dtype: int64

Labeled train rows: 600000 / 800000 (75.0%)

Class balance (labeled only):
 0.0    0.5
1.0    0.5
Name: proportion, dtype: float64


In [8]:
# %%
# --- Cell 5: feature ranges ---
# Only look at labeled rows, since those are what Day 21 will actually train on
X_labeled = X_train[labeled_mask]

feature_stats = pd.DataFrame({
    "min":  X_labeled.min(axis=0),
    "max":  X_labeled.max(axis=0),
    "mean": X_labeled.mean(axis=0),
    "std":  X_labeled.std(axis=0),
})

print(feature_stats.describe())   # summary of the summary — spread across all 2381 columns

# flag columns with an unusually huge range — worth a second look, not necessarily a bug
wide_range = feature_stats[(feature_stats["max"] - feature_stats["min"]) > 1e6]
print(f"\n{len(wide_range)} columns have a range > 1,000,000")

                min           max          mean           std
count  2.381000e+03  2.381000e+03  2.381000e+03  2.381000e+03
mean  -6.710640e+06  5.513359e+07  5.744252e+05  6.076156e+05
std    1.107956e+08  4.276512e+08  2.758142e+07  1.102896e+07
min   -4.278190e+09  0.000000e+00 -4.187928e+05  0.000000e+00
25%   -5.400000e+01  9.000161e-01 -4.913333e-03  1.310213e-02
50%   -1.300000e+01  1.600000e+01  1.542654e-03  2.731584e-01
75%    0.000000e+00  7.700000e+01  1.113500e-02  8.921849e-01
max    5.120000e+02  4.294967e+09  1.345846e+09  5.021095e+08

138 columns have a range > 1,000,000


In [9]:
# %%
# --- Cell 6: missing / degenerate values ---
nan_count = np.isnan(X_labeled).sum()
inf_count = np.isinf(X_labeled).sum()
print(f"NaN values in X_labeled: {nan_count}")
print(f"Inf values in X_labeled: {inf_count}")

# columns that are constant (zero variance) carry no information for a classifier
zero_variance_cols = (feature_stats["std"] == 0).sum()
print(f"Zero-variance columns: {zero_variance_cols}")



NaN values in X_labeled: 0
Inf values in X_labeled: 0
Zero-variance columns: 40


In [10]:
# %%
# --- Cell 7: metadata sanity check ---
print(metadata_df.head())
print("\nMetadata label counts:\n", metadata_df["label"].value_counts())

                                              sha256 appeared  label avclass  \
0  0abb4fda7d5b13801d63bee53e5e256be43e141faa077a...  2006-12      0     NaN   
1  c9cafff8a596ba8a80bafb4ba8ae6f2ef3329d95b85f15...  2007-01      0     NaN   
2  eac8ddb4970f8af985742973d6f0e06902d42a3684d791...  2007-02      0     NaN   
3  7f513818bcc276c531af2e641c597744da807e21cc1160...  2007-02      0     NaN   
4  ca65e1c387a4cc9e7d8a8ce12bf1bcf9f534c9032b9d95...  2007-02      0     NaN   

  subset  
0  train  
1  train  
2  train  
3  train  
4  train  

Metadata label counts:
 label
 0    400000
 1    400000
-1    200000
Name: count, dtype: int64
